# Mortgage Rate Enrichment
Enrich both the combined sold and listings datasets by merging in the national 30-year fixed mortgage rate from the St. Louis Federal Reserve (FRED). This introduces interns to fetching live economic data from a public API and performing a time-series join on a monthly key.

# Objective
Fetch the FRED MORTGAGE30US series, resample it from weekly to monthly frequency, and merge it onto both combined datasets using a year-month key derived from transaction dates.

# Background
The FRED MORTGAGE30US series is published weekly (every Thursday) by Freddie Mac via the St. Louis Federal Reserve. Because MLS transactions are analyzed by calendar month, interns must resample the weekly data to a monthly average before joining. The data can be fetched directly from FRED as a CSV—no API key required.

# Step 1 – Fetch the mortgage rate data from FRED

In [45]:
import pandas as pd

url = r"MORTGAGE30US.csv"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

# Step 2 – Resample weekly rates to monthly averages

In [63]:
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
    mortgage.groupby('year_month')['rate_30yr_fixed']
    .mean()
    .reset_index()
)

# Step 3 – Create a matching year_month key on the MLS datasets

In [64]:
mortgage_monthly['year_month'] = mortgage_monthly['year_month'].astype('str')

In [65]:
mortgage_monthly[['year', 'month']] = mortgage_monthly['year_month'].str.split('-', n=1, expand=True)

# Step 4 – Merge

In [51]:
sold_path = r'week_2_final_deliverables\sold_data.csv'
listing_path = r'week_2_final_deliverables\listing_data.csv'
sold = pd.read_csv(sold_path)
listings = pd.read_csv(listing_path)

C:\Users\elizm\AppData\Local\Temp\ipykernel_34584\3184123764.py:3: DtypeWarning: Columns (25: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv(sold_path)
C:\Users\elizm\AppData\Local\Temp\ipykernel_34584\3184123764.py:4: DtypeWarning: Columns (23: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv(listing_path)


In [70]:
sold['month'] = sold['month'].astype('int64')
listings['month'] = listings['month'].astype('int64')
mortgage_monthly['month'] = mortgage_monthly['month'].astype('int64')
mortgage_monthly['year'] = mortgage_monthly['year'].astype('int64')

In [72]:
sold_with_rates = sold.merge(mortgage_monthly, on=['year','month'], how='left')
listings_with_rates = listings.merge(mortgage_monthly, on=['year','month'], how='left')

# Step 5 – Validate the merge

In [73]:
sold_with_rates.drop(columns='year_month', inplace=True)
listings_with_rates.drop(columns='year_month', inplace=True)

In [74]:
# Check for any unmatched rows (rate should not be null)
print(sold_with_rates['rate_30yr_fixed'].isnull().sum())
print(listings_with_rates['rate_30yr_fixed'].isnull().sum())

0
0


# Preview

In [75]:
print(
    sold_with_rates[
        ['CloseDate', 'year','month', 'ClosePrice', 'rate_30yr_fixed']
    ].head()
)

    CloseDate  year  month  ClosePrice  rate_30yr_fixed
0  2024-02-07  2024      2   1249000.0            6.776
1  2024-02-15  2024      2    374000.0            6.776
2  2024-02-16  2024      2    300000.0            6.776
3  2024-02-06  2024      2      9000.0            6.776
4  2024-02-02  2024      2    730000.0            6.776


In [77]:
sold_target_path = r'week_3_final_deliverables\sold_enriched.csv'
listings_target_path = r'week_3_final_deliverables\listings_enriched.csv'
sold_with_rates.to_csv(sold_target_path)
listings_with_rates.to_csv(listings_target_path)